# Environment and data audit

Standalone, clean-kernel execution with checkpointed prerequisites. Synthetic mode is explicit. See docs/RESEARCH_PROTOCOL.md for scientific definitions. Final test access is guarded.

In [ ]:
import os

if os.environ.get("BRFSS_SYNTHETIC", "0") != "1":
    from google.colab import drive

    drive.mount("/content/drive")
    os.environ.setdefault(
        "BRFSS_REPO",
        "/content/drive/MyDrive/BRFSS2015_Statistical_Methods_AZ/BRFSS2015-Statistical-Methods",
    )

In [ ]:
import os
import sys
from pathlib import Path

REPO = Path(os.environ.get("BRFSS_REPO", Path.cwd()))
if not (REPO / "pyproject.toml").exists():
    REPO = REPO.parent
if not (REPO / "pyproject.toml").exists():
    raise FileNotFoundError("Set BRFSS_REPO to the extracted repository folder.")
os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
SYNTHETIC = os.environ.get("BRFSS_SYNTHETIC", "0") == "1"
OUT = Path(
    os.environ.get(
        "BRFSS_OUT",
        "runs/notebook_synthetic"
        if SYNTHETIC
        else "/content/drive/MyDrive/BRFSS2015_Statistical_Methods_AZ/runs/primary",
    )
)
DATA_PATH = os.environ.get("BRFSS_DATA")
CODEBOOK_PATH = os.environ.get("BRFSS_CODEBOOK")
PROFILE = os.environ.get("BRFSS_PROFILE", "quick")
MODE = os.environ.get("BRFSS_MODE", "binary")

In [ ]:
if not SYNTHETIC:
    from google.colab import drive

    drive.mount("/content/drive")
    import subprocess

    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
from scripts.verify_environment import verify

versions = verify()

In [ ]:
from src.pipeline import run

run(
    OUT,
    data=DATA_PATH,
    codebook=CODEBOOK_PATH,
    synthetic=SYNTHETIC,
    n=960,
    profile=PROFILE,
    mode=MODE,
    through="audit",
)

In [ ]:
import json

artifact = json.loads((OUT / "audit.json").read_text())
print(json.dumps(artifact, indent=2, default=str)[:5000])